[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/13_Human_in_the_Loop.ipynb)

# DiveLab

## Notebook 13 — Human-in-the-Loop Buoyancy Control

**Guiding question:** What changes when the controller is explicitly modeled as a human diver?

Up to now, DiveLab has used mathematical controllers as simplified models of diver decision-making.

In this notebook, we make the human element explicit.

We introduce:

- perception thresholds;
- reaction time;
- intermittent decisions;
- overcorrection;
- bounded actions;
- anticipation.

The goal is to study control behavior, not to prescribe real-world diving procedures.

## Learning objectives

By the end of this lab, you will be able to:

- model the diver as part of the feedback loop;
- distinguish continuous and intermittent control;
- represent perception thresholds with a deadband;
- include human reaction delay;
- simulate overcorrection;
- compare different control styles;
- interpret skilled behavior as better estimation, anticipation and action timing;
- connect human control with delay, MPC and estimation.

# 1. The human controller

A diver does not continuously solve differential equations.

Instead, control may look more like:

1. observe;
2. estimate what is happening;
3. decide whether correction is needed;
4. act;
5. wait and observe the response;
6. correct again.

This is an **intermittent feedback loop**.

## The loop

```text
physical diver
      |
      v
perception / instruments
      |
      v
internal estimate
      |
      v
decision
      |
      v
action
      |
      +------------------> physical diver
```

The controller is no longer an abstract equation alone.

It is a model of human perception and action.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 2. Nonlinear diver model

We reuse the nonlinear vertical model from previous notebooks.

In [ ]:
rho = 1025.0
g = 9.80665
P0 = 101325.0

mass = 90.0
Cd = 0.9
A_drag = 0.7

z_e = 20.0
Vs_e = 0.005

def pressure_at_depth(z):
    return P0 + rho * g * z

def gas_volume_at_depth(z, surface_volume):
    return surface_volume * P0 / pressure_at_depth(z)

Vg_e = gas_volume_at_depth(z_e, Vs_e)
fixed_volume = mass / rho - Vg_e

def buoyant_force(z, Vs):
    return rho * g * (
        fixed_volume + gas_volume_at_depth(z, Vs)
    )

def drag_force(v):
    return 0.5 * rho * Cd * A_drag * v * abs(v)

def acceleration(z, v, Vs, disturbance=0.0):
    return (
        buoyant_force(z, Vs)
        - mass * g
        - drag_force(v)
        + disturbance
    ) / mass

# 3. Ideal continuous controller

As a reference, define a continuous controller:

$$
u
=
K_z(z-z_e)
-
K_vv
-
K_V(V_s-V_{s,e}).
$$

This controller reacts at every integration step.

A human does not.

In [ ]:
Kz = 8e-5
Kv = 8e-4
Kv_s = 0.25

u_max = 0.00035

def ideal_controller(z, v, Vs):
    u = (
        Kz * (z - z_e)
        - Kv * v
        - Kv_s * (Vs - Vs_e)
    )
    return np.clip(u, -u_max, u_max)

# 4. Human feature 1 — perception threshold

Small errors may not trigger a correction.

This can be modeled with a **deadband**.

If:

$$
|e_z|<\Delta z_{\text{perception}}
$$

and:

$$
|v|<\Delta v_{\text{perception}},
$$

the diver takes no action.

In [ ]:
depth_deadband = 0.15
velocity_deadband = 0.03

def inside_deadband(z, v):
    return (
        abs(z - z_e) < depth_deadband
        and abs(v) < velocity_deadband
    )

A deadband can be useful because it prevents constant reaction to tiny fluctuations.

But a deadband that is too large may allow the unstable plant to drift too far before correction begins.

# 5. Human feature 2 — reaction delay

The diver notices an error, but action is not instantaneous.

Let:

$$
\tau_r
$$

be the reaction delay.

The action at time $t$ may depend on a perceived state from:

$$
t-\tau_r.
$$

# 6. Human feature 3 — intermittent action

A human controller may act only at discrete decision times.

For example:

- observe continuously;
- update the control decision every 0.5 s;
- hold the chosen action until the next decision.

This is a form of **sample-and-hold control**.

In [ ]:
decision_interval = 0.5  # seconds
reaction_delay = 0.6     # seconds

# 7. Human feature 4 — overcorrection

A common control pattern is to apply too much correction after waiting too long.

We model this with a gain multiplier:

$$
g_h.
$$

For:

$$
g_h>1,
$$

the action is more aggressive than the baseline controller.

In [ ]:
human_gain_multiplier = 1.0

# 8. Human controller model

The controller will:

- react only at decision times;
- use a delayed state;
- ignore very small errors;
- hold the chosen action between decisions;
- optionally scale the correction.

In [ ]:
def human_decision(z, v, Vs, gain_multiplier=1.0):
    if inside_deadband(z, v):
        return 0.0

    u = gain_multiplier * (
        Kz * (z - z_e)
        - Kv * v
        - Kv_s * (Vs - Vs_e)
    )

    return np.clip(u, -u_max, u_max)

# 9. Simulator with a human controller

In [ ]:
def simulate_human_controller(
    z0=z_e,
    v0=0.08,
    Vs0=Vs_e,
    duration=40.0,
    dt=0.01,
    reaction_delay=0.6,
    decision_interval=0.5,
    gain_multiplier=1.0,
    disturbance_fn=None,
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    z = np.zeros(n)
    v = np.zeros(n)
    Vs = np.zeros(n)
    u = np.zeros(n)

    z[0] = z0
    v[0] = v0
    Vs[0] = Vs0

    delay_steps = int(round(reaction_delay / dt))
    decision_steps = max(1, int(round(decision_interval / dt)))

    held_u = 0.0

    for k in range(n - 1):

        if k % decision_steps == 0:
            j = max(0, k - delay_steps)

            held_u = human_decision(
                z[j],
                v[j],
                Vs[j],
                gain_multiplier=gain_multiplier
            )

        u[k] = held_u

        disturbance = 0.0 if disturbance_fn is None else disturbance_fn(t[k])

        a = acceleration(
            z[k],
            v[k],
            Vs[k],
            disturbance
        )

        v[k + 1] = v[k] + a * dt
        z[k + 1] = max(z[k] - v[k + 1] * dt, 0.0)
        Vs[k + 1] = max(Vs[k] + held_u * dt, 0.0)

    u[-1] = u[-2]

    return t, z, v, Vs, u

# 10. Compare ideal and human-in-the-loop control

In [ ]:
def simulate_ideal(
    z0=z_e,
    v0=0.08,
    Vs0=Vs_e,
    duration=40.0,
    dt=0.01
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    z = np.zeros(n)
    v = np.zeros(n)
    Vs = np.zeros(n)
    u = np.zeros(n)

    z[0] = z0
    v[0] = v0
    Vs[0] = Vs0

    for k in range(n - 1):
        uk = ideal_controller(z[k], v[k], Vs[k])
        u[k] = uk

        a = acceleration(z[k], v[k], Vs[k])

        v[k + 1] = v[k] + a * dt
        z[k + 1] = max(z[k] - v[k + 1] * dt, 0.0)
        Vs[k + 1] = max(Vs[k] + uk * dt, 0.0)

    u[-1] = u[-2]

    return t, z, v, Vs, u

t_i, z_i, v_i, Vs_i, u_i = simulate_ideal()
t_h, z_h, v_h, Vs_h, u_h = simulate_human_controller()

In [ ]:
plt.plot(t_i, z_i, label="Ideal continuous controller")
plt.plot(t_h, z_h, label="Human-in-the-loop model")
plt.axhline(z_e, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Continuous vs intermittent delayed control")
plt.grid(True)
plt.legend()
plt.show()

The human model may recover more slowly or show larger oscillations because:

- information is delayed;
- actions are intermittent;
- small errors may be ignored;
- corrections are held between decisions.

# 11. Control actions are intermittent

In [ ]:
plt.plot(t_h, u_h)

plt.xlabel("Time [s]")
plt.ylabel("Control input u")
plt.title("Sample-and-hold human control action")
plt.grid(True)
plt.show()

Instead of continuously varying control, the action appears in steps.

This captures the idea:

> decide → act → wait → reassess.

# 12. Perception threshold experiment

Compare several deadband sizes.

In [ ]:
deadbands = [0.05, 0.15, 0.40]
results_deadband = {}

original_depth_deadband = depth_deadband

for db in deadbands:
    depth_deadband = db
    results_deadband[db] = simulate_human_controller()

depth_deadband = original_depth_deadband

for db in deadbands:
    t_db, z_db, v_db, Vs_db, u_db = results_deadband[db]
    plt.plot(t_db, z_db, label=f"deadband = {db:.2f} m")

plt.axhline(z_e, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Effect of perception threshold")
plt.grid(True)
plt.legend()
plt.show()

A larger deadband means fewer corrections.

That may reduce unnecessary activity, but it also allows more drift before action begins.

# 13. Reaction-delay experiment

In [ ]:
delays = [0.1, 0.6, 1.5]

for tau in delays:
    t_tau, z_tau, v_tau, Vs_tau, u_tau = simulate_human_controller(
        reaction_delay=tau
    )
    plt.plot(t_tau, z_tau, label=f"delay = {tau:.1f} s")

plt.axhline(z_e, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Human reaction delay")
plt.grid(True)
plt.legend()
plt.show()

This reconnects directly with Notebook 05.

Longer reaction delay can lead to:

- later corrections;
- larger deviations;
- overcorrection;
- oscillation.

# 14. Decision-frequency experiment

Compare a diver who updates decisions frequently with one who waits longer between corrections.

In [ ]:
intervals = [0.2, 0.5, 1.2]

for interval in intervals:
    t_int, z_int, v_int, Vs_int, u_int = simulate_human_controller(
        decision_interval=interval
    )
    plt.plot(t_int, z_int, label=f"decision every {interval:.1f} s")

plt.axhline(z_e, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Effect of intermittent decision timing")
plt.grid(True)
plt.legend()
plt.show()

Intermittent control can work well if decisions are timely and appropriately scaled.

But long pauses allow an unstable plant more time to evolve between corrections.

# 15. Overcorrection experiment

Now vary the human gain multiplier.

In [ ]:
gains = [0.6, 1.0, 1.8]

for gh in gains:
    t_g, z_g, v_g, Vs_g, u_g = simulate_human_controller(
        gain_multiplier=gh
    )
    plt.plot(t_g, z_g, label=f"gain multiplier = {gh:.1f}")

plt.axhline(z_e, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Under-correction and overcorrection")
plt.grid(True)
plt.legend()
plt.show()

Small gain may produce weak correction.

Large gain may produce excessive correction and oscillatory behavior.

This is the human analogue of controller tuning.

# 16. A simple novice-like model

For teaching purposes, define a hypothetical novice-like controller with:

- longer reaction delay;
- larger deadband;
- less frequent decisions;
- stronger corrective action after delay.

This is not a claim about real individual divers.

It is a synthetic control profile used to study dynamics.

In [ ]:
def simulate_novice_like():
    global depth_deadband

    old_db = depth_deadband
    depth_deadband = 0.35

    result = simulate_human_controller(
        reaction_delay=1.2,
        decision_interval=1.0,
        gain_multiplier=1.6
    )

    depth_deadband = old_db
    return result

# 17. A simple experienced-like model

Define a synthetic experienced-like controller with:

- shorter effective reaction delay;
- smaller but nonzero deadband;
- more frequent reassessment;
- moderate correction gain.

Again, this is a conceptual model, not a measurement of real expertise.

In [ ]:
def simulate_experienced_like():
    global depth_deadband

    old_db = depth_deadband
    depth_deadband = 0.10

    result = simulate_human_controller(
        reaction_delay=0.3,
        decision_interval=0.3,
        gain_multiplier=0.9
    )

    depth_deadband = old_db
    return result

In [ ]:
t_n, z_n, v_n, Vs_n, u_n = simulate_novice_like()
t_e, z_eh, v_eh, Vs_eh, u_eh = simulate_experienced_like()

plt.plot(t_n, z_n, label="Synthetic novice-like profile")
plt.plot(t_e, z_eh, label="Synthetic experienced-like profile")
plt.axhline(z_e, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Two hypothetical human-control profiles")
plt.grid(True)
plt.legend()
plt.show()

The difference is not "good person vs bad person".

It is a difference in modeled control characteristics:

- perception;
- timing;
- anticipation;
- action size;
- reassessment frequency.

# 18. Add a disturbance

Now apply a temporary disturbance and compare the two profiles.

In [ ]:
def disturbance(t):
    if 10.0 <= t <= 12.0:
        return 20.0
    return 0.0

In [ ]:
def run_profile_with_disturbance(
    reaction_delay,
    decision_interval,
    gain_multiplier,
    deadband
):
    global depth_deadband

    old_db = depth_deadband
    depth_deadband = deadband

    result = simulate_human_controller(
        reaction_delay=reaction_delay,
        decision_interval=decision_interval,
        gain_multiplier=gain_multiplier,
        disturbance_fn=disturbance
    )

    depth_deadband = old_db
    return result

In [ ]:
t_nd, z_nd, v_nd, Vs_nd, u_nd = run_profile_with_disturbance(
    reaction_delay=1.2,
    decision_interval=1.0,
    gain_multiplier=1.6,
    deadband=0.35
)

t_ed, z_ed, v_ed, Vs_ed, u_ed = run_profile_with_disturbance(
    reaction_delay=0.3,
    decision_interval=0.3,
    gain_multiplier=0.9,
    deadband=0.10
)

plt.plot(t_nd, z_nd, label="Novice-like profile")
plt.plot(t_ed, z_ed, label="Experienced-like profile")
plt.axhline(z_e, linestyle="--")
plt.axvspan(10, 12, alpha=0.15)

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Human control after a disturbance")
plt.grid(True)
plt.legend()
plt.show()

# 19. Skill as prediction, not just faster reaction

The more interesting interpretation of skill is not simply:

> faster reaction.

A skilled diver may also:

- recognize trends earlier;
- estimate velocity better;
- anticipate buoyancy changes;
- make smaller corrections;
- wait for the plant response before correcting again.

This connects human skill to:

- state estimation;
- delay management;
- predictive control.

# 20. Anticipatory human control

We can add a simple anticipation term.

Instead of using only current velocity $v$, estimate a short future depth:

$$
z_{\text{pred}}
\approx
z-vT_p.
$$

Then react partly to the predicted future error.

In [ ]:
prediction_time = 1.5
K_predict = 4e-5

def predictive_human_decision(z, v, Vs, gain_multiplier=1.0):
    z_pred = z - v * prediction_time

    current_error = z - z_e
    predicted_error = z_pred - z_e

    if (
        abs(current_error) < depth_deadband
        and abs(v) < velocity_deadband
        and abs(predicted_error) < depth_deadband
    ):
        return 0.0

    u = gain_multiplier * (
        Kz * current_error
        + K_predict * predicted_error
        - Kv * v
        - Kv_s * (Vs - Vs_e)
    )

    return np.clip(u, -u_max, u_max)

This is not full MPC.

It is a simple model of the thought:

> "If I continue moving like this, where will I be shortly?"

# 21. Simulator with predictive human decision

In [ ]:
def simulate_predictive_human(
    z0=z_e,
    v0=0.08,
    Vs0=Vs_e,
    duration=40.0,
    dt=0.01,
    reaction_delay=0.3,
    decision_interval=0.3,
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    z = np.zeros(n)
    v = np.zeros(n)
    Vs = np.zeros(n)
    u = np.zeros(n)

    z[0] = z0
    v[0] = v0
    Vs[0] = Vs0

    delay_steps = int(round(reaction_delay / dt))
    decision_steps = max(1, int(round(decision_interval / dt)))

    held_u = 0.0

    for k in range(n - 1):

        if k % decision_steps == 0:
            j = max(0, k - delay_steps)

            held_u = predictive_human_decision(
                z[j], v[j], Vs[j]
            )

        u[k] = held_u

        a = acceleration(z[k], v[k], Vs[k])

        v[k + 1] = v[k] + a * dt
        z[k + 1] = max(z[k] - v[k + 1] * dt, 0.0)
        Vs[k + 1] = max(Vs[k] + held_u * dt, 0.0)

    u[-1] = u[-2]

    return t, z, v, Vs, u

In [ ]:
t_base, z_base, v_base, Vs_base, u_base = simulate_experienced_like()
t_pred, z_pred, v_pred, Vs_pred, u_pred = simulate_predictive_human()

plt.plot(t_base, z_base, label="Reactive human model")
plt.plot(t_pred, z_pred, label="Predictive human model")
plt.axhline(z_e, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Reactive vs anticipatory human control")
plt.grid(True)
plt.legend()
plt.show()

Anticipation can reduce the need for large late corrections.

This creates a direct bridge from human-in-the-loop behavior back to Notebook 12 and MPC.

# 22. Human control as hybrid dynamics

The human-controlled system is no longer just a smooth differential equation.

It combines:

- continuous physical dynamics;
- discrete decisions;
- delayed information;
- threshold-triggered events;
- held control actions.

This is a form of **hybrid dynamical system**.

## Continuous part

The diver moves continuously:

$$
\dot z=-v
$$

$$
\dot v=f(z,v,V_s).
$$

## Discrete part

At decision times:

$$
u^+=\phi(\text{perceived state}).
$$

Between decisions:

$$
u(t)=\text{constant}.
$$

So the complete system mixes continuous evolution and discrete events.

# 23. Why this matters

Human behavior introduces dynamics that are absent from a purely continuous controller:

- reaction delay;
- intermittent updates;
- threshold effects;
- memory;
- anticipation;
- changing control strategy.

This helps explain why human-in-the-loop systems can show rich behavior even when the physical plant is simple.

# 24. Performance metrics for human control

We can quantify synthetic control profiles using:

- maximum depth deviation;
- RMS depth error;
- maximum velocity;
- total control activity.

These are not measures of real diver skill.

They are metrics for comparing model behavior.

In [ ]:
def performance_metrics(t, z, v, u):
    depth_error = z - z_e

    rmse = np.sqrt(np.mean(depth_error**2))
    max_depth_error = np.max(np.abs(depth_error))
    max_speed = np.max(np.abs(v))

    dt_local = t[1] - t[0]
    control_activity = np.sum(np.abs(u)) * dt_local

    return {
        "RMSE depth": rmse,
        "Max depth error": max_depth_error,
        "Max speed": max_speed,
        "Control activity": control_activity,
    }

In [ ]:
novice_metrics = performance_metrics(t_n, z_n, v_n, u_n)
experienced_metrics = performance_metrics(t_e, z_eh, v_eh, u_eh)

print("Synthetic novice-like profile")
for key, value in novice_metrics.items():
    print(f"{key:20s}: {value:.4f}")

print()
print("Synthetic experienced-like profile")
for key, value in experienced_metrics.items():
    print(f"{key:20s}: {value:.4f}")

# 25. A systems view of training

This control perspective suggests a useful conceptual decomposition of skill development.

Training can improve:

### Perception

Detect relevant changes earlier.

### Estimation

Infer velocity and trend more accurately.

### Timing

Correct before error becomes large.

### Gain selection

Use smaller, better-scaled actions.

### Prediction

Anticipate future buoyancy and motion.

### Patience

Allow the plant to respond before adding another correction.

This is a control-theory interpretation of skill acquisition.

# 26. Important limitation

Human behavior is much more complex than this model.

Real control depends on:

- attention;
- workload;
- stress;
- task goals;
- visual references;
- proprioception;
- vestibular information;
- instrument use;
- equipment familiarity;
- environment.

The models in this notebook are deliberately synthetic.

They are useful for reasoning about feedback, not for predicting an individual diver.

# 27. Connect the full DiveLab story

We can now interpret the system as:

```text
physical dynamics
      |
      v
sensors + perception
      |
      v
state estimation
      |
      v
human prediction
      |
      v
decision logic
      |
      v
intermittent action
      |
      v
physical dynamics
```

Every previous notebook fits somewhere in this loop.

# Exercises

### 1. Increase reaction delay

Try:

```python
reaction_delay = 2.0
```

What changes?

### 2. Increase the deadband

Try:

```python
depth_deadband = 0.5
```

Does the system drift farther before correction?

### 3. Increase decision interval

Try decisions every:

```python
1.5
```

seconds.

How does intermittent control interact with instability?

### 4. Overcorrection

Try:

```python
gain_multiplier = 2.5
```

Does stronger action improve or degrade behavior?

### 5. Prediction horizon

Change:

```python
prediction_time
```

How much anticipation helps before model error becomes important?

# Challenge — adaptive human controller

Design a controller that changes its own behavior based on recent performance.

For example:

- if oscillation grows, reduce gain;
- if error persists, increase gain slightly;
- if velocity trend becomes large, shorten the decision interval.

This introduces the idea of **adaptive control**.

In [ ]:
# Your code here

# Summary

In this notebook we made the diver explicitly part of the control model.

We introduced:

- perception deadband;
- reaction delay;
- intermittent decision-making;
- sample-and-hold actions;
- overcorrection;
- synthetic novice-like and experienced-like profiles;
- anticipatory control;
- hybrid dynamics.

### Core insight

> **Human buoyancy control is not continuous perfect feedback. It is delayed, intermittent, perception-limited and anticipatory.**

A skilled control strategy can be interpreted as improving:

$$
\boxed{
\text{perception}
+
\text{estimation}
+
\text{timing}
+
\text{prediction}
+
\text{action scaling}
}
$$

### Next

Notebook 14 can introduce **adaptive control and learning**:

> Can the diver-controller improve its own parameters from repeated experience?

That would connect control theory with learning and adaptation.